# Evo 2 Texture Hypothesis Test

Tests whether Evo 2's embedding geometry tracks **statistical texture** (GC content,
k-mer frequencies) or **higher-order biological structure**.

## Hypothesis

If Evo 2 is primarily a texture detector, sequences with matching GC/dinucleotide
statistics should cluster together in embedding space regardless of true biological
function.

## Four Sequence Conditions

| Condition | Description |
|-----------|-------------|
| **Real chr22** | Authentic human chromosome 22 sequences |
| **Random** | Uniform random DNA (A/C/G/T equiprobable) |
| **Texture-matched** | GC-matched random sequences (same GC% as real) |
| **Dinucleotide-shuffled** | Shuffled preserving dinucleotide frequencies |

Geometric stability differences across conditions reveal the degree to which
Evo 2 responds to low-order sequence statistics vs. higher-order structure.

## Setup Instructions

1. Upload `evaluation_harness.py` to `/content/`
2. Set GPU runtime to **A100** (7B model)
3. Run cells in order

---

In [ ]:
# Install Dependencies
#
# LOCAL (7B only): evo2 package (Vortex) with return_embeddings=True
# No API -- this notebook tests only evo2_7b_base on A100.

print('Installing core dependencies...')
!pip install -q shesha-geometry matplotlib seaborn pandas scipy requests

# Install ninja (build tool for flash-attn)
!pip install -q ninja
import sys, os

# Pin torch to 2.7.1+cu128 -- Arc Institute recommended version.
print('Pinning torch to 2.7.1+cu128 (Arc Institute recommended)...')
!pip install -q torch==2.7.1 --index-url https://download.pytorch.org/whl/cu128

import torch
print(f'torch {torch.__version__} | CUDA {torch.version.cuda}')

# Install flash-attn 2.8.0.post2 from source (matches torch 2.7.1+cu128)
print('Building flash-attn 2.8.0.post2 from source (~10-15 min)...')
!pip install flash-attn==2.8.0.post2 --no-build-isolation

!pip install -q evo2

print(f'torch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU count: {torch.cuda.device_count()}')

from evo2 import Evo2
print('\nEvo2 package loaded successfully')
print('Done!')

In [ ]:
# Configuration
#
# This notebook tests the TEXTURE HYPOTHESIS:
#   "Evo 2's RC success on real DNA is driven by statistical texture
#    (GC content + k-mer frequencies), not by training distribution
#    overlap or higher-order biological structure."
#
# Four sequence conditions:
#   1. Real chr22      -- in-distribution, full biological structure
#   2. Pure random      -- uniform base sampling, no texture
#   3. Texture-matched  -- random seqs sampled to match chr22 k-mer distribution
#   4. Dinuc-shuffled   -- real chr22 shuffled at dinucleotide level
#                         (preserves k-mer freqs, destroys higher-order structure)
#
# Only evo2_7b_base (7B, 8K context) tested locally on A100.

PHASE = 'full'

OUTPUT_BASE = './results/evo2_texture_hypothesis/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

import os
import sys
sys.path.insert(0, '.')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

CHECKPOINT = 'evo2_7b_base'
CHECKPOINT_LABEL = 'Evo2-7B-8K'
EMBEDDING_LAYER = 'blocks.28.mlp.l3'
BATCH_SIZE = 4

CONFIG = {
    'quick': {
        'n_sequences': 500,
        'seq_length': 1000,
        'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10],
    },
    'full': {
        'n_sequences': 10000,
        'seq_length': 1000,
        'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10],
    },
}

config = CONFIG[PHASE]
print(f'Phase: {PHASE.upper()}')
print(f'Checkpoint: {CHECKPOINT_LABEL} (local Vortex)')
print(f'Sequences per condition: {config["n_sequences"]}')
print(f'Sequence length: {config["seq_length"]} bp')
print(f'Embedding layer: {EMBEDDING_LAYER}')
print(f'Results: {RESULTS_DIR}')
print(f'Cache: {CACHE_DIR}')

In [ ]:
# Fetch Real chr22 Sequences from UCSC Genome Browser API

import numpy as np
import requests
import json

DNA_BASES = ['A', 'C', 'G', 'T']
COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}


def fetch_chr22_sequences(n_sequences, seq_length, seed=320):
    """
    Fetch real human chr22 sequences from UCSC Genome Browser API (hg38).
    Downloads large contiguous chunks (2 MB each) in a few API calls,
    then windows locally. Much faster than one-at-a-time fetching.
    Filters out N-containing windows.
    """
    cache_path = f'{CACHE_DIR}/real_chr22_{n_sequences}x{seq_length}.json'
    if os.path.exists(cache_path):
        print(f'Loading cached chr22 sequences from {cache_path}')
        with open(cache_path, 'r') as f:
            sequences = json.load(f)
        print(f'  Loaded {len(sequences)} sequences')
        return sequences[:n_sequences]

    rng = np.random.default_rng(seed)
    # chr22 euchromatic region: ~16.5M to ~50.8M (hg38)
    chr22_start = 16_500_000
    chr22_end = 50_800_000
    api_base = 'https://api.genome.ucsc.edu/getData/sequence'

    # Fetch large contiguous chunks and window locally.
    # Each chunk is 2 MB; the euchromatic arm is ~34 MB, so ~17 chunks
    # covers it entirely. This takes seconds, not hours.
    chunk_size = 2_000_000
    chunk_starts = list(range(chr22_start, chr22_end, chunk_size))
    rng.shuffle(chunk_starts)  # randomize order for sampling diversity

    print(f'Fetching chr22 in {len(chunk_starts)} bulk chunks (2 MB each)...')
    full_dna = {}  # start -> dna string
    for ci, cs in enumerate(chunk_starts):
        ce = min(cs + chunk_size, chr22_end)
        try:
            resp = requests.get(
                api_base,
                params={'genome': 'hg38', 'chrom': 'chr22',
                        'start': cs, 'end': ce},
                timeout=120,
            )
            resp.raise_for_status()
            data = resp.json()
            dna = data.get('dna', '').upper()
            if len(dna) > 0:
                full_dna[cs] = dna
            print(f'  Chunk {ci+1}/{len(chunk_starts)}: '
                  f'{cs:,}-{ce:,} ({len(dna):,} bp)', end='\r')
        except Exception as e:
            print(f'  Chunk {ci+1} failed: {e}')
    print()

    # Window locally: sample random positions within fetched chunks
    sequences = []
    all_chunks = sorted(full_dna.keys())
    attempts = 0
    max_attempts = n_sequences * 5
    while len(sequences) < n_sequences and attempts < max_attempts:
        # Pick a random chunk, then a random offset within it
        cs = rng.choice(all_chunks)
        dna = full_dna[cs]
        if len(dna) < seq_length:
            attempts += 1
            continue
        offset = rng.integers(0, len(dna) - seq_length)
        window = dna[offset:offset + seq_length]
        if len(window) == seq_length and all(b in 'ACGT' for b in window):
            sequences.append(window)
            if len(sequences) % 2000 == 0:
                print(f'  Windowed {len(sequences)}/{n_sequences}')
        attempts += 1

    if len(sequences) < n_sequences:
        print(f'  WARNING: only extracted {len(sequences)}/{n_sequences} '
              f'clean windows (N-regions filtered)')

    # Cache to Drive
    with open(cache_path, 'w') as f:
        json.dump(sequences, f)
    print(f'  Cached {len(sequences)} sequences to {cache_path}')
    return sequences


real_sequences = fetch_chr22_sequences(config['n_sequences'], config['seq_length'])
print(f'\nReal chr22: {len(real_sequences)} sequences')
print(f'Sample: {real_sequences[0][:60]}...')

In [ ]:
# Generate All Four Sequence Conditions
#
# 1. Real chr22           -- already fetched
# 2. Pure random          -- uniform base sampling (existing baseline)
# 3. Texture-matched      -- random seqs matching chr22 k-mer distribution
# 4. Dinucleotide-shuffled -- real chr22 shuffled preserving dinuc freqs

import numpy as np
from collections import Counter
from itertools import product as itertools_product

rng_gen = np.random.default_rng(320)


# --- Condition 2: Pure Random ---
def generate_pure_random(n_sequences, seq_length, rng):
    """Uniform i.i.d. base sampling."""
    return [
        ''.join(rng.choice(DNA_BASES, size=seq_length))
        for _ in range(n_sequences)
    ]


# --- Condition 3: Texture-Matched Synthetics ---
def compute_kmer_freqs(sequences, k=2):
    """Compute k-mer frequency distribution across a set of sequences."""
    kmers = [''.join(combo) for combo in itertools_product('ACGT', repeat=k)]
    total_counts = Counter()
    for seq in sequences:
        for i in range(len(seq) - k + 1):
            kmer = seq[i:i+k]
            if all(b in 'ACGT' for b in kmer):
                total_counts[kmer] += 1
    total = sum(total_counts.values())
    freqs = {km: total_counts.get(km, 0) / total for km in kmers}
    return freqs


def generate_texture_matched(n_sequences, seq_length, target_dinuc_freqs, rng):
    """
    Generate random sequences that match the dinucleotide frequency
    distribution of a target (real chr22). Uses a first-order Markov
    chain parameterized by the observed dinucleotide transition matrix.

    This preserves GC content and dinucleotide frequencies but destroys
    all higher-order structure (genes, repeats, evolutionary history).

    Caveat: first-order Markov matches dinucleotide but NOT trinucleotide
    frequencies exactly. If the verdict comes back "partial" (30-70%
    recovery), one explanation would be that Evo 2 is sensitive to
    trinucleotide or higher-order texture specifically. In that case,
    a second-order Markov chain (trinucleotide transitions) should be
    tested as a follow-up. The dinuc-shuffled condition partially
    controls for this since it preserves exact per-sequence dinuc
    counts from real DNA.
    """
    # Build transition matrix from dinucleotide freqs
    bases = list('ACGT')
    base_idx = {b: i for i, b in enumerate(bases)}

    # Marginal base frequencies (from dinucleotide freqs)
    marginal = {b: 0.0 for b in bases}
    for dinuc, freq in target_dinuc_freqs.items():
        marginal[dinuc[0]] += freq

    # Transition probabilities: P(next_base | current_base)
    trans = np.zeros((4, 4))
    for dinuc, freq in target_dinuc_freqs.items():
        i = base_idx[dinuc[0]]
        j = base_idx[dinuc[1]]
        trans[i, j] = freq
    # Normalize rows
    row_sums = trans.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    trans = trans / row_sums

    # Marginal as initial distribution
    init_probs = np.array([marginal[b] for b in bases])
    init_probs = init_probs / init_probs.sum()

    sequences = []
    for _ in range(n_sequences):
        seq = []
        # First base from marginal
        curr = rng.choice(4, p=init_probs)
        seq.append(bases[curr])
        for _ in range(seq_length - 1):
            curr = rng.choice(4, p=trans[curr])
            seq.append(bases[curr])
        sequences.append(''.join(seq))
    return sequences


# --- Condition 4: Dinucleotide-Shuffled Real Sequences ---
def dinucleotide_shuffle(sequence, rng):
    """
    Shuffle a DNA sequence while preserving exact dinucleotide frequencies.
    Uses the Altschul-Erickson algorithm (Euler path on the dinucleotide
    graph). This preserves the exact dinucleotide counts of the original
    sequence but destroys all higher-order structure.

    Simplified approach: build an Eulerian path through the dinucleotide
    graph. If the graph is not Eulerian (edge cases), fall back to a
    greedy edge-shuffling approach.
    """
    if len(sequence) <= 2:
        return sequence

    bases = list('ACGT')
    base_idx = {b: i for i, b in enumerate(bases)}

    # Build edge lists for each node (base)
    edges = {b: [] for b in bases}
    for i in range(len(sequence) - 1):
        b1, b2 = sequence[i], sequence[i + 1]
        if b1 in base_idx and b2 in base_idx:
            edges[b1].append(b2)

    # Shuffle outgoing edges for each node
    for b in bases:
        rng.shuffle(edges[b])

    # Build Eulerian path starting from the first base
    start = sequence[0]
    stack = [start]
    path = []
    edge_idx = {b: 0 for b in bases}

    while stack:
        v = stack[-1]
        if v in edges and edge_idx[v] < len(edges[v]):
            next_v = edges[v][edge_idx[v]]
            edge_idx[v] += 1
            stack.append(next_v)
        else:
            path.append(stack.pop())

    path.reverse()

    # The Eulerian path should have length = len(sequence)
    # If not (due to disconnected graph), pad with the last base
    result = ''.join(path)
    if len(result) < len(sequence):
        result = result + sequence[-1] * (len(sequence) - len(result))
    return result[:len(sequence)]


def generate_dinuc_shuffled(real_seqs, rng):
    """Dinucleotide-shuffle each real sequence independently."""
    return [dinucleotide_shuffle(s, rng) for s in real_seqs]


# === Generate all conditions ===

print('Computing chr22 dinucleotide frequencies...')
chr22_dinuc_freqs = compute_kmer_freqs(real_sequences, k=2)
print('  Dinucleotide freqs:', {k: f'{v:.4f}' for k, v in sorted(chr22_dinuc_freqs.items())})

gc_content = np.mean([sum(1 for b in s if b in 'GC') / len(s) for s in real_sequences])
print(f'  Mean GC content: {gc_content:.4f}')

print(f'\nGenerating pure random sequences ({config["n_sequences"]})...')
random_sequences = generate_pure_random(
    config['n_sequences'], config['seq_length'], rng_gen
)

print(f'Generating texture-matched synthetics ({config["n_sequences"]})...')
texture_sequences = generate_texture_matched(
    config['n_sequences'], config['seq_length'],
    chr22_dinuc_freqs, rng_gen
)

print(f'Generating dinucleotide-shuffled real ({config["n_sequences"]})...')
shuffled_sequences = generate_dinuc_shuffled(real_sequences, rng_gen)

# Validation: check that texture-matched and shuffled preserve dinuc freqs
print('\n--- Validation ---')
for label, seqs in [
    ('Real chr22', real_sequences),
    ('Pure random', random_sequences),
    ('Texture-matched', texture_sequences),
    ('Dinuc-shuffled', shuffled_sequences),
]:
    gc = np.mean([sum(1 for b in s if b in 'GC') / len(s) for s in seqs])
    dinucs = compute_kmer_freqs(seqs, k=2)
    trinucs = compute_kmer_freqs(seqs, k=3)
    print(f'  {label:20s}: GC={gc:.4f}, '
          f'top dinuc={max(dinucs, key=dinucs.get)}={dinucs[max(dinucs, key=dinucs.get)]:.4f}')

CONDITIONS = {
    'real_chr22':        ('Real chr22',          real_sequences),
    'pure_random':       ('Pure Random',         random_sequences),
    'texture_matched':   ('Texture-Matched',     texture_sequences),
    'dinuc_shuffled':    ('Dinuc-Shuffled Real',  shuffled_sequences),
}

print(f'\n4 conditions ready, {config["n_sequences"]} sequences each')


In [ ]:
# K-mer Frequency Analysis (K-mer Frequency Analysis
#
# For each condition, compute GC content and dinucleotide/trinucleotide
# frequency vectors for forward sequences and their RCs.
# Show cosine similarity between forward and RC frequency vectors.
# This grounds the texture claim quantitatively.

import numpy as np
from scipy.spatial.distance import cosine


def reverse_complement(sequence):
    return ''.join(COMPLEMENT.get(b, b) for b in reversed(sequence))


def kmer_frequency_vector(sequences, k=2):
    """Return a numpy vector of k-mer frequencies in canonical order."""
    kmers = [''.join(combo) for combo in itertools_product('ACGT', repeat=k)]
    freqs = compute_kmer_freqs(sequences, k=k)
    return np.array([freqs.get(km, 0.0) for km in kmers])


print('K-MER FREQUENCY ANALYSIS: Forward vs. Reverse Complement')
print('=' * 70)
print(f'{"Condition":25s} {"GC_fwd":>8s} {"GC_rc":>8s} '
      f'{"dinuc_cos":>10s} {"trinuc_cos":>11s}')
print('-' * 70)

kmer_results = []

for cond_key, (cond_label, seqs) in CONDITIONS.items():
    rc_seqs = [reverse_complement(s) for s in seqs]

    # GC content
    gc_fwd = np.mean([sum(1 for b in s if b in 'GC') / len(s) for s in seqs])
    gc_rc = np.mean([sum(1 for b in s if b in 'GC') / len(s) for s in rc_seqs])

    # Dinucleotide frequency vectors
    dinuc_fwd = kmer_frequency_vector(seqs, k=2)
    dinuc_rc = kmer_frequency_vector(rc_seqs, k=2)
    dinuc_cos = 1.0 - cosine(dinuc_fwd, dinuc_rc)

    # Trinucleotide frequency vectors
    trinuc_fwd = kmer_frequency_vector(seqs, k=3)
    trinuc_rc = kmer_frequency_vector(rc_seqs, k=3)
    trinuc_cos = 1.0 - cosine(trinuc_fwd, trinuc_rc)

    print(f'{cond_label:25s} {gc_fwd:8.4f} {gc_rc:8.4f} '
          f'{dinuc_cos:10.6f} {trinuc_cos:11.6f}')

    kmer_results.append({
        'condition': cond_key,
        'label': cond_label,
        'gc_forward': gc_fwd,
        'gc_rc': gc_rc,
        'dinuc_cosine_fwd_rc': dinuc_cos,
        'trinuc_cosine_fwd_rc': trinuc_cos,
    })

print('\nNote: RC preserves GC content by definition (A<->T, C<->G).')
print('High cosine similarity = k-mer texture is preserved under RC.')
print('This is necessary but not sufficient to explain Evo 2 RC behavior.')

In [ ]:
# Perturbation Suite (RC-focused)
#
# For each condition: compute embeddings for forward + RC.
# Also include SNP perturbations for calibration.

from dataclasses import dataclass, field


def mutate_dna(sequence, mutation_rate, rng):
    """SNP mutations at the given rate."""
    seq = list(sequence)
    n_mutations = max(1, int(len(seq) * mutation_rate))
    positions = rng.choice(len(seq), size=n_mutations, replace=False)
    for pos in positions:
        original = seq[pos]
        alternatives = [b for b in DNA_BASES if b != original]
        seq[pos] = rng.choice(alternatives)
    return ''.join(seq)


@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: list
    params: dict = field(default_factory=dict)
    description: str = ''


class TexturePerturbationSuite:
    """Perturbation suite focused on RC + calibration SNPs."""
    def __init__(self, seed=320, snp_rates=None):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.05, 0.10]

    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f'snp_{int(rate * 100)}pct'
            perturbed = [mutate_dna(s, rate, self.rng) for s in sequences]
            results[name] = PerturbedSet(
                name=name, category='snp', sequences=perturbed,
                params={'mutation_rate': rate},
                description=f'SNP mutations at {rate*100:.0f}% rate',
            )
        results['reverse_complement'] = PerturbedSet(
            name='reverse_complement', category='rc',
            sequences=[reverse_complement(s) for s in sequences],
            params={}, description='Reverse complement transformation',
        )
        return results


suite = TexturePerturbationSuite(seed=320, snp_rates=config['snp_rates'])
print('Perturbation suite ready')
print(f'  SNP rates: {config["snp_rates"]}')
print(f'  + reverse_complement')

In [ ]:
# Evo 2 Model Wrapper (Local 7B only)

import torch
import gc
from evo2 import Evo2


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        mem = torch.cuda.memory_allocated() / 1e9
        print(f'  GPU memory after cleanup: {mem:.2f} GB')


def load_evo2_local(checkpoint_name, batch_size=4):
    """Load Evo 2 locally via Vortex. Return embedding function."""
    print(f'[LOCAL] Loading {checkpoint_name} via Vortex...')
    device = 'cuda:0'

    evo2_model = Evo2(checkpoint_name)
    n_params = sum(p.numel() for p in evo2_model.model.parameters()) / 1e6

    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f'  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB')

    layer_name = EMBEDDING_LAYER
    print(f'  Embedding layer: {layer_name}')

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_total = len(sequences)
        for idx, seq in enumerate(sequences):
            if (idx + 1) % 50 == 0 or idx == 0 or idx == n_total - 1:
                print(f'    Seq {idx+1}/{n_total}', end='\r')
            input_ids = torch.tensor(
                evo2_model.tokenizer.tokenize(seq), dtype=torch.int,
            ).unsqueeze(0).to(device)
            _, layer_embs = evo2_model(
                input_ids, return_embeddings=True, layer_names=[layer_name],
            )
            emb = layer_embs[layer_name]  # (1, seq_len, hidden_dim)
            pooled = emb.mean(dim=1).squeeze(0)  # (hidden_dim,)
            embeddings.append(pooled.cpu().float().numpy())
        print()
        return np.array(embeddings, dtype=np.float32)

    return embed, evo2_model, n_params


print('Evo 2 local wrapper ready')

In [ ]:
# Evaluation Harness

from evaluation_harness import StabilityHarness

harness = StabilityHarness(
    window_size=0,
    metric='cosine',
    n_splits=30,
    seed=320,
    max_samples=2500,
    n_bootstrap=config['n_bootstrap'],
)

print(f'Shesha StabilityHarness configured (bootstrap={config["n_bootstrap"]})')

In [ ]:
# Run Experiment -- All Four Conditions
#
# Load Evo 2 7B once, then run all four conditions sequentially.
# Each condition: embed forward, embed all perturbations, evaluate.

import time
import pandas as pd
from evaluation_harness import ModelReport

print('Loading Evo 2 7B...')
embed_fn, evo2_model_obj, n_params_m = load_evo2_local(CHECKPOINT, BATCH_SIZE)

print('\n' + '=' * 70)
print('EVO 2 TEXTURE HYPOTHESIS TEST')
print('=' * 70)

all_detailed_rows = []

for cond_idx, (cond_key, (cond_label, cond_seqs)) in enumerate(CONDITIONS.items()):
    print(f"\n{'='*70}")
    print(f'[{cond_idx+1}/4] Condition: {cond_label} ({cond_key})')
    print(f'  Sequences: {len(cond_seqs)}')
    print('=' * 70)

    start_time = time.time()

    # Generate perturbations for this condition
    # Use a fresh suite per condition with offset seed so SNP mutation
    # positions are independent across conditions
    # Offset seed per condition so SNP mutation positions are independent
    # across conditions (avoids coupling calibration numbers).
    cond_seed = 320 + cond_idx * 1000
    cond_suite = TexturePerturbationSuite(seed=cond_seed, snp_rates=config['snp_rates'])
    perturbed_sets = cond_suite.run_all(cond_seqs)

    # Cache paths
    cache_clean = f'{CACHE_DIR}/texture_{cond_key}_clean.npy'

    # Clean embeddings
    if os.path.exists(cache_clean):
        print(f'Loading cached clean embeddings: {cache_clean}')
        embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings ({len(cond_seqs)} sequences)...')
        embeddings_clean = embed_fn(cond_seqs)
        np.save(cache_clean, embeddings_clean)
        print(f'  Cached to {cache_clean}')
    print(f'  Shape: {embeddings_clean.shape}')

    # Perturbed embeddings
    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/texture_{cond_key}_{pert_name}.npy'
        if os.path.exists(cache_pert):
            print(f'  Loading cached: {pert_name}')
            perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])

    # Run Shesha evaluation
    print('\nRunning Shesha evaluation...')
    report = harness.evaluate_all_perturbations(
        model_name=f'{CHECKPOINT_LABEL}_{cond_key}',
        embeddings_clean=embeddings_clean,
        perturbed_dict=perturbed_embeddings,
    )

    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({
            'condition': cond_key,
            'condition_label': cond_label,
            'checkpoint': CHECKPOINT,
            'label': CHECKPOINT_LABEL,
            'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0),
            'rdm_drift': metrics.get('rdm_drift_score', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0),
            'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0),
        })

    elapsed = time.time() - start_time
    summary = report.summary()
    print(f'\nCompleted {cond_label} in {elapsed/60:.1f} min')
    print(f'  Composite Stability: {summary["mean_composite_stability"]:.4f}')
    print(f'  RDM Similarity:      {summary["mean_rdm_similarity_score"]:.4f}')

# Free model
del evo2_model_obj
cleanup_gpu()

# Save results
df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/evo2_texture_{PHASE}_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nAll results saved to {detailed_path}')
print(df_detailed.to_string(index=False))

print('\n' + '=' * 70)
print('EXPERIMENT COMPLETE')
print('=' * 70)


In [ ]:
# Key Result -- RC Stability Across Conditions
#
# This is the decisive comparison. If the texture hypothesis is correct:
#   - Real chr22:        HIGH RC stability (known from prior experiments)
#   - Pure random:       LOW RC stability  (known from prior experiments)
#   - Texture-matched:   HIGH RC stability (CONFIRMS texture hypothesis)
#   - Dinuc-shuffled:    HIGH RC stability (k-mer texture sufficient)
#
# If texture hypothesis is WRONG:
#   - Texture-matched:   LOW RC stability (texture not sufficient)
#   - Dinuc-shuffled:    could go either way

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'font.size': 12})

# Extract RC rows
df_rc = df_detailed[df_detailed['perturbation'] == 'reverse_complement'].copy()

# Order conditions for display
cond_order = ['real_chr22', 'dinuc_shuffled', 'texture_matched', 'pure_random']
cond_labels = {
    'real_chr22': 'Real chr22\n(in-distribution)',
    'dinuc_shuffled': 'Dinuc-Shuffled\n(k-mer preserved)',
    'texture_matched': 'Texture-Matched\n(Markov synthetic)',
    'pure_random': 'Pure Random\n(uniform i.i.d.)',
}
cond_colors = {
    'real_chr22': '#16A34A',
    'dinuc_shuffled': '#0891B2',
    'texture_matched': '#7C3AED',
    'pure_random': '#DC2626',
}

fig, axes = plt.subplots(1, 3, figsize=(17, 6))

for col_idx, (metric, ylabel, title) in enumerate([
    ('composite', 'Composite Stability', 'A. Composite Stability (RC)'),
    ('rdm_similarity', 'RDM Similarity', 'B. RDM Similarity (RC)'),
    ('pert_magnitude', 'Perturbation Magnitude', 'C. Perturbation Magnitude (RC)'),
]):
    ax = axes[col_idx]
    values = []
    labels = []
    colors = []
    for ck in cond_order:
        row = df_rc[df_rc['condition'] == ck]
        if len(row) > 0:
            values.append(row[metric].values[0])
            labels.append(cond_labels[ck])
            colors.append(cond_colors[ck])

    bars = ax.bar(range(len(values)), values, color=colors, edgecolor='black',
                  linewidth=0.8, alpha=0.85)
    ax.set_xticks(range(len(values)))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.2, axis='y')

    # Add value labels
    for i, v in enumerate(values):
        ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

fig.suptitle('Evo 2 (7B): RC Stability Across Sequence Conditions\n'
             'Does Statistical Texture Explain the Real/Synthetic RC Gap?',
             fontsize=14, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/evo2_texture_rc_comparison_{PHASE}.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(f'{RESULTS_DIR}/evo2_texture_rc_comparison_{PHASE}.pdf',
            bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved to {RESULTS_DIR}/evo2_texture_rc_comparison_{PHASE}.png')

In [ ]:
# Full Per-Perturbation Heatmap (All Conditions x All Perturbations)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap: Composite Stability
pivot_comp = df_detailed.pivot_table(
    values='composite', index='perturbation', columns='condition_label',
)
# Reorder columns
col_order = [cond_labels[ck].replace('\n', ' ') for ck in cond_order
             if CONDITIONS[ck][0] in pivot_comp.columns]
col_order_raw = [CONDITIONS[ck][0] for ck in cond_order
                 if CONDITIONS[ck][0] in pivot_comp.columns]
pivot_comp = pivot_comp[col_order_raw]

ax = axes[0]
sns.heatmap(pivot_comp, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=0.5,
            ax=ax, linewidths=0.5, cbar_kws={'label': 'Composite'})
ax.set_title('Composite Stability', fontweight='bold')
ax.set_xlabel('Sequence Condition')
ax.set_ylabel('Perturbation')

# Heatmap: RDM Similarity
pivot_rdm = df_detailed.pivot_table(
    values='rdm_similarity', index='perturbation', columns='condition_label',
)
pivot_rdm = pivot_rdm[col_order_raw]

ax = axes[1]
sns.heatmap(pivot_rdm, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
            ax=ax, linewidths=0.5, cbar_kws={'label': 'RDM Similarity'})
ax.set_title('RDM Similarity', fontweight='bold')
ax.set_xlabel('Sequence Condition')
ax.set_ylabel('Perturbation')

fig.suptitle('Evo 2 (7B): Texture Hypothesis -- Full Perturbation Breakdown',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/evo2_texture_heatmap_{PHASE}.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved to {RESULTS_DIR}/evo2_texture_heatmap_{PHASE}.png')

In [ ]:
# K-mer Texture Similarity Visualization
#
# Show the k-mer frequency vectors (dinuc + trinuc) for each condition,
# and the cosine similarity between forward and RC.

import pandas as pd

df_kmer = pd.DataFrame(kmer_results)
print('K-mer Frequency Analysis Summary:')
print(df_kmer.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Dinucleotide frequency comparison (real chr22 vs texture-matched)
ax = axes[0]
dinucs_real = compute_kmer_freqs(real_sequences, k=2)
dinucs_texture = compute_kmer_freqs(texture_sequences, k=2)
dinucs_random = compute_kmer_freqs(random_sequences, k=2)
dinucs_shuffled = compute_kmer_freqs(shuffled_sequences, k=2)

sorted_keys = sorted(dinucs_real.keys())
x_pos = np.arange(len(sorted_keys))
width = 0.2

ax.bar(x_pos - 1.5*width, [dinucs_real[k] for k in sorted_keys],
       width, label='Real chr22', color='#16A34A', alpha=0.8)
ax.bar(x_pos - 0.5*width, [dinucs_shuffled[k] for k in sorted_keys],
       width, label='Dinuc-shuffled', color='#0891B2', alpha=0.8)
ax.bar(x_pos + 0.5*width, [dinucs_texture[k] for k in sorted_keys],
       width, label='Texture-matched', color='#7C3AED', alpha=0.8)
ax.bar(x_pos + 1.5*width, [dinucs_random[k] for k in sorted_keys],
       width, label='Pure random', color='#DC2626', alpha=0.8)

ax.set_xticks(x_pos)
ax.set_xticklabels(sorted_keys, fontsize=7, rotation=45)
ax.set_ylabel('Frequency')
ax.set_title('A. Dinucleotide Frequency Distributions', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2, axis='y')

# Panel B: Forward vs RC cosine similarity by condition
ax = axes[1]
cond_names = [r['label'] for r in kmer_results]
dinuc_sims = [r['dinuc_cosine_fwd_rc'] for r in kmer_results]
trinuc_sims = [r['trinuc_cosine_fwd_rc'] for r in kmer_results]

x_pos = np.arange(len(cond_names))
ax.bar(x_pos - 0.15, dinuc_sims, 0.3, label='Dinucleotide',
       color='#2563EB', alpha=0.85)
ax.bar(x_pos + 0.15, trinuc_sims, 0.3, label='Trinucleotide',
       color='#EA580C', alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(cond_names, fontsize=9)
ax.set_ylabel('Cosine Similarity (Fwd vs RC)')
ax.set_title('B. K-mer Preservation Under RC', fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0.95, 1.005)
ax.grid(True, alpha=0.2, axis='y')

# Add value labels
for i, (d, t) in enumerate(zip(dinuc_sims, trinuc_sims)):
    ax.text(i - 0.15, d + 0.001, f'{d:.4f}', ha='center', fontsize=7)
    ax.text(i + 0.15, t + 0.001, f'{t:.4f}', ha='center', fontsize=7)

fig.suptitle('K-mer Texture Analysis: All Four Conditions',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/evo2_texture_kmer_analysis_{PHASE}.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved to {RESULTS_DIR}/evo2_texture_kmer_analysis_{PHASE}.png')

In [ ]:
# Save K-mer Analysis Results

df_kmer.to_csv(f'{RESULTS_DIR}/evo2_texture_kmer_analysis_{PHASE}.csv', index=False)
print(f'K-mer analysis saved to {RESULTS_DIR}/evo2_texture_kmer_analysis_{PHASE}.csv')

# Save a combined summary
summary_rows = []
for cond_key in cond_order:
    rc_row = df_rc[df_rc['condition'] == cond_key]
    kmer_row = [r for r in kmer_results if r['condition'] == cond_key]
    if len(rc_row) > 0 and len(kmer_row) > 0:
        summary_rows.append({
            'condition': cond_key,
            'label': kmer_row[0]['label'],
            'gc_content': kmer_row[0]['gc_forward'],
            'dinuc_cos_fwd_rc': kmer_row[0]['dinuc_cosine_fwd_rc'],
            'trinuc_cos_fwd_rc': kmer_row[0]['trinuc_cosine_fwd_rc'],
            'rc_composite': rc_row['composite'].values[0],
            'rc_rdm_similarity': rc_row['rdm_similarity'].values[0],
            'rc_pert_magnitude': rc_row['pert_magnitude'].values[0],
        })

df_summary = pd.DataFrame(summary_rows)
summary_path = f'{RESULTS_DIR}/evo2_texture_summary_{PHASE}.csv'
df_summary.to_csv(summary_path, index=False)
print(f'Summary saved to {summary_path}')
print()
print(df_summary.to_string(index=False))